# Cross-modal QC: HepG2 Cell Painting vs. scRNA-seq

Quality-control checks for the compounds shared between EU-OPENSCREEN HepG2
Cell Painting (`morphology_umap_analysis.ipynb`) and Tahoe `hepg2_cells.h5ad`,
before the cross-modal integration in `morphology_expression_integration.ipynb`.

Pipeline:

1. Match the 120 expression drug names to the morphology compound reference.
2. Normalize / feature-select HepG2 morphology wells; run cell-level
   expression PCA (needed for plate-level expression profiles).
3. **Percent replicating** (Way et al. 2022) between plates for both
   modalities.
4. Morphology consensus profiles + leave-one-out agreement.
5. Phenotypic activity vs. DMSO (**copairs** mAP) for morphology.

In [2]:
import glob
import os
import re
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from pycytominer import feature_select, normalize
from pycytominer.cyto_utils import infer_cp_features
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")
sc.settings.verbosity = 0
plt.rcParams["figure.facecolor"] = "#fcfcfb"
plt.rcParams["axes.facecolor"] = "#fcfcfb"
plt.rcParams["font.size"] = 10

RANDOM_STATE = 0
N_PCS = 30

PALETTE = [
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4",
    "#008300", "#4a3aa7", "#e34948", "#6b4226", "#38a5b0",
    "#a24fbf", "#c2914e", "#5a6bd6", "#c65da0", "#6f9e1f",
    "#d65f5f", "#3f7d8c", "#9a7dcb", "#b25b2a", "#4f9d6e",
    "#8c8c8c",
]


## 1. Load gene expression

`hepg2_cells.h5ad` is raw UMI counts (sparse, `float32`). `drug` and
`moa-fine` are per-cell categorical annotations; there is no vehicle/DMSO
control group in this file, so expression profiles below are compared to
each other, not to an untreated baseline. The morphology side is different:
the raw per-well annotation files do carry 784 DMSO negative-control wells
per HepG2 site (`Metadata_EOS == "DMSO"`), silently dropped so far because
`load_well_pdid` below only resolves wells to a compound `pdid` and DMSO has
none - section 10 recovers them for a negative-control-based activity check.


In [3]:
# C:\Users\nbrouwer1\Local projects\cell-painting\data\WS4_data\TAHOE single_cell_level
rna = ad.read_h5ad("../../data/WS4_data/TAHOE single_cell_level/hepg2_cells.h5ad")
rna.var_names = rna.var["gene_symbol"].astype(str)
rna.var_names_make_unique()

print(f"{rna.n_obs} cells x {rna.n_vars} genes")
print(f"{rna.obs['drug'].nunique()} drugs, {rna.obs['plate'].nunique()} plates, "
      f"{rna.obs['moa-fine'].nunique()} moa-fine classes, cell line {rna.obs['cell_line_id'].unique().tolist()}")

drug_summary = (
    rna.obs.groupby("drug", observed=True)
    .agg(n_cells=("drug", "size"), moa_fine=("moa-fine", "first"))
    .sort_values("n_cells")
)
drug_summary.head()


389085 cells x 62710 genes
120 drugs, 14 plates, 21 moa-fine classes, cell line ['CVCL_1098']


,n_cells,moa_fine
drug,,
Menadione,925,unclear
Clofarabine,1006,DNA synthesis/repair inhibitor
Pemetrexed,1015,DNA synthesis/repair inhibitor
Canagliflozin,1079,Glucose transporter inhibitor
Silodosin,1110,unclear


## 2. Match compounds between datasets

The morphology library's compound reference
(`annotations/pd_export_..._compounds_standardized.csv`) gives a canonical
`name` plus a `;`-separated `synonyms` list per `pdid`. Match each
`hepg2_cells.h5ad` drug name against that reference in three passes: exact
name, then synonym, then the same two checks after stripping a trailing
salt/hydrate/formulation suffix - either parenthetical (e.g. `"Idarubicin
(hydrochloride)"` -> `"Idarubicin"`) or a bare trailing salt-form word (e.g.
`"Larotrectinib sulfate"` -> `"Larotrectinib"`, `"Ralimetinib dimesylate"` ->
`"Ralimetinib"`) - these free-text name differences are the main source of
missed matches, not real compound identity differences. This resolves all
120 drugs to a `pdid` - a perfect match.


In [4]:
ANNOTATION_DIR = "../../data/WS4_data/OpenScreen morphology-20260901T075256Z-1-001/OpenScreen morphology/annotations/annotations"

compounds = pd.read_csv(os.path.join(ANNOTATION_DIR, "pd_export_04_2022_2464_compounds_standardized.csv"))
compounds = compounds.drop_duplicates(subset="name", keep="first")
name_to_pdid = dict(zip(compounds["name"].str.lower().str.strip(), compounds["pdid"]))

synonym_to_pdid = {}
for _, row in compounds.iterrows():
    for syn in str(row["synonyms"] or "").split(";"):
        syn = syn.strip().lower()
        if syn:
            synonym_to_pdid.setdefault(syn, row["pdid"])

# Common salt/hydrate/counter-ion suffixes that show up as a bare trailing
# word rather than inside parentheses (e.g. "Larotrectinib sulfate",
# "Ralimetinib dimesylate") - stripped repeatedly in case more than one is
# stacked (e.g. "... dihydrochloride hydrate").
SALT_SUFFIX_RE = re.compile(
    r"\s+(?:hydrochlorides?|dihydrochloride|trihydrochloride|sulfates?|disulfate|"
    r"bisulfate|mesylate|dimesylate|tosylate|besylate|maleate|fumarate|"
    r"citrate|acetate|phosphate|diphosphate|hydrobromide|dihydrobromide|"
    r"succinate|oxalate|nitrate|lactate|malate|gluconate|tartrate|bitartrate|"
    r"hemihydrate|monohydrate|dihydrate|trihydrate|tetrahydrate|hydrate|"
    r"sodium|potassium|calcium|iodide|bromide|chloride|palmitate|stearate|"
    r"benzoate|carbonate|bicarbonate|pamoate|isethionate|napadisylate|"
    r"edisylate)\s*$",
    re.IGNORECASE,
)


def strip_formulation_suffix(name: str) -> str:
    name = re.sub(r"\s*\([^)]*\)\s*$", "", name).strip()
    prev = None
    while prev != name:
        prev = name
        name = SALT_SUFFIX_RE.sub("", name).strip()
    return name


def match_drug_to_pdid(drug: str) -> str | None:
    key = drug.lower().strip()
    for candidate in (key, strip_formulation_suffix(key)):
        if candidate in name_to_pdid:
            return name_to_pdid[candidate]
        if candidate in synonym_to_pdid:
            return synonym_to_pdid[candidate]
    return None


drug_to_pdid = {d: match_drug_to_pdid(d) for d in rna.obs["drug"].cat.categories}
n_matched = sum(v is not None for v in drug_to_pdid.values())
print(f"Matched {n_matched}/{len(drug_to_pdid)} drugs to the morphology compound reference")
print("Unmatched:", [d for d, p in drug_to_pdid.items() if p is None])

Matched 120/120 drugs to the morphology compound reference
Unmatched: []


## 3. Gene expression: cell-level PCA, averaged per drug

Standard scanpy pipeline (total-count normalize, log1p, highly-variable
genes, PCA) at single-cell resolution, then average each drug's cells'
PC coordinates into one embedding per drug. Averaging post-PCA (rather than
pseudobulking raw counts pre-PCA) keeps the gene-expression side on the same
"reduce fine-grained observations to one PCA space, then average within
group" logic used for morphology below.


In [5]:
sc.pp.normalize_total(rna)
sc.pp.log1p(rna)
sc.pp.highly_variable_genes(rna, n_top_genes=2000)
sc.pp.pca(rna, n_comps=N_PCS, mask_var="highly_variable", random_state=RANDOM_STATE)

rna_pcs = pd.DataFrame(rna.obsm["X_pca"], index=rna.obs_names)
rna_pcs["drug"] = rna.obs["drug"].to_numpy()
rna_compound_pca = rna_pcs.groupby("drug", observed=True).mean()

moa_by_drug = rna.obs.groupby("drug", observed=True)["moa-fine"].first()

print(f"Per-drug expression embedding: {rna_compound_pca.shape}")
rna_compound_pca.head()


Per-drug expression embedding: (120, 30)


,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
drug,,,,,,,,,,,,,,,,,,,,,
18β-Glycyrrhetinic acid,0.302304,-0.545984,0.356316,-0.089024,-0.013785,-0.025536,0.243104,0.087827,-0.006344,0.075287,...,-0.129787,-0.041849,0.015050,0.048321,0.031882,-0.023476,0.031685,0.047019,0.102022,0.010792
5-Fluorouracil,-0.366180,0.799737,-0.228654,0.322678,-0.415608,-0.253918,-0.181348,-0.281235,-0.104727,-0.251689,...,0.032546,0.110452,-0.002552,-0.022824,-0.001909,-0.061364,-0.100724,-0.005887,0.066062,0.040238
8-Hydroxyquinoline,1.003545,-0.496191,0.224861,-0.049724,-0.011082,0.076248,0.203890,0.204085,-0.090487,-0.195492,...,0.010113,-0.026299,0.024974,-0.079174,-0.006926,-0.016738,-0.024223,-0.126444,-0.037333,-0.049592
AT7519,0.267017,-0.213019,0.323521,0.073948,0.050305,0.052901,0.051460,0.188226,0.110703,0.054173,...,-0.092122,-0.004392,0.024753,0.087732,-0.050142,0.041367,0.082586,0.120788,0.053169,-0.002172
AZD-8055,0.182385,0.002434,-0.053235,-0.115544,0.012579,0.151400,-0.018940,0.336889,0.063267,0.420353,...,0.149069,-0.159854,-0.029057,-0.049904,-0.130794,0.064152,0.030732,0.099636,0.154699,-0.069731


## 4. Morphology: load, normalize, and feature-select HepG2 wells

Reuses the same loading / per-plate `mad_robustize` normalization / well-
annotation join as `morphology_umap_analysis.ipynb`, restricted to the four
HepG2 production sites (`FMP_U2OS` is excluded - the gene expression data is
HepG2-only). Compound-level PCA for integration is left to
`morphology_expression_integration.ipynb`; here we only need well-level
features for percent replicating, consensus, and DMSO activity.

In [6]:
# abspath: CWD + relative path exceeds Windows MAX_PATH before `..` collapses
DATA_DIR = os.path.abspath(
    "../../data/WS4_data/OpenScreen morphology-20260901T075256Z-1-001/"
    "OpenScreen morphology/aggregated_data/aggregated_data/"
)
HEPG2_SITES = [d for d in sorted(os.listdir(DATA_DIR)) if d.endswith("HepG2")]
COMPARTMENTS = ["Cells", "Nuc", "Cyto"]

PLATE_RE = re.compile(r"B\d{3,}")
REPLICATE_RE = re.compile(r"(?<![A-Za-z])R\d+")


def parse_plate_replicate(filename: str) -> tuple[str, str]:
    return PLATE_RE.search(filename).group(), REPLICATE_RE.search(filename).group()


def load_site_profiles(site_dir: str, site_name: str) -> pd.DataFrame:
    frames = []
    for f in sorted(glob.glob(os.path.join(site_dir, "*.csv"))):
        d = pd.read_csv(os.path.abspath(f))
        feature_cols = [c for c in d.columns if not c.startswith("Metadata_")]
        d[feature_cols] = d[feature_cols].astype("float32")
        plate, replicate = parse_plate_replicate(os.path.basename(f))
        d["Metadata_Site"] = site_name
        d["Metadata_PlateID"] = plate
        d["Metadata_Replicate"] = replicate
        d["Metadata_SourceFile"] = f"{site_name}_{plate}_{replicate}"
        frames.append(d)
    return pd.concat(frames, ignore_index=True)


def normalize_per_plate(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    parts = [
        normalize(
            group, features=features, meta_features="infer",
            samples="all", method="mad_robustize",
        )
        for _, group in df.groupby("Metadata_SourceFile")
    ]
    return pd.concat(parts, ignore_index=True)


print(f"HepG2 sites: {HEPG2_SITES}")
hepg2_raw = {site: load_site_profiles(os.path.join(DATA_DIR, site), site) for site in HEPG2_SITES}

FEATURES = infer_cp_features(next(iter(hepg2_raw.values())), compartments=COMPARTMENTS)
hepg2_normalized = {site: normalize_per_plate(df, FEATURES) for site, df in hepg2_raw.items()}
hepg2_pooled = pd.concat(hepg2_normalized.values(), ignore_index=True)
print(f"{len(hepg2_pooled)} wells pooled across {len(HEPG2_SITES)} HepG2 sites, {len(FEATURES)} inferred features")


HepG2 sites: ['FMP_HepG2', 'IMTM_HepG2', 'MEDINA_HepG2', 'USC_HepG2']
42847 wells pooled across 4 HepG2 sites, 2977 inferred features


In [7]:
FEATURE_SELECT_OPS = [
    "variance_threshold", "frequency_threshold", "correlation_threshold",
    "drop_na_columns", "blocklist", "drop_outliers",
]
hepg2_selected = feature_select(hepg2_pooled, features=FEATURES, operation=FEATURE_SELECT_OPS)
morph_feature_cols = [c for c in hepg2_selected.columns if not c.startswith("Metadata_")]
print(f"{len(morph_feature_cols)} features selected on pooled HepG2 data")

SITE_ANNOTATION_FILES = {
    "FMP_HepG2": "2022-07-08_Annotation_Bioactives_HepG2.csv",
    "IMTM_HepG2": "2023-08-14_Annotation2_IMTM_HepG2.csv",
    "MEDINA_HepG2": "2023-11-28_Annotation_MEDINA_HepG2.csv",
    "USC_HepG2": "2023-11-28_Annotation_USC_HepG2.csv",
}
eos_to_pdid = pd.read_csv(os.path.join(ANNOTATION_DIR, "2024-08-02_EOS_pdid.csv"))
eos_to_pdid_map = dict(zip(eos_to_pdid["Metadata_EOS"], eos_to_pdid["Metadata_pdid"]))


def load_well_pdid(site: str) -> pd.DataFrame:
    ann = pd.read_csv(os.path.join(ANNOTATION_DIR, SITE_ANNOTATION_FILES[site]))
    ann = ann.rename(columns={"Metadata_Plate": "Metadata_PlateID", "Metadata_Batch": "Metadata_Replicate"})
    ann["Metadata_pdid"] = ann["Metadata_EOS"].map(eos_to_pdid_map)
    ann["Metadata_negcon"] = ann["Metadata_EOS"] == "DMSO"
    return ann[["Metadata_PlateID", "Metadata_Well", "Metadata_Replicate", "Metadata_pdid", "Metadata_negcon"]]


hepg2_selected = pd.concat(
    [
        group.merge(load_well_pdid(site), on=["Metadata_PlateID", "Metadata_Well", "Metadata_Replicate"], how="left")
        for site, group in hepg2_selected.groupby("Metadata_Site")
    ],
    ignore_index=True,
)
print(f"{hepg2_selected['Metadata_pdid'].notna().mean():.1%} of HepG2 wells resolved to a pdid")


636 features selected on pooled HepG2 data
91.6% of HepG2 wells resolved to a pdid


## 5. Shared compound set and well-level export table

Intersect expression drugs with compounds that have usable HepG2 morphology
wells after feature selection. Build `morph_export_df` (shared compounds only)
used by the percent-replicating and consensus checks below.

In [8]:
# Compounds with both an expression label and at least one HepG2 morphology well.
pdid_to_drug = {pdid: drug for drug, pdid in drug_to_pdid.items() if pdid is not None}
morph_drugs = (
    hepg2_selected["Metadata_pdid"].map(pdid_to_drug).dropna().astype(str).unique()
)
rna_drugs = set(rna.obs["drug"].astype(str).unique())
shared_compounds = sorted(set(morph_drugs) & rna_drugs)
moa_by_drug = rna.obs.groupby("drug", observed=True)["moa-fine"].first()
print(f"{len(shared_compounds)} compounds shared between morphology wells and expression")

morph_export_df = hepg2_selected.copy()
morph_export_df["Metadata_Drug"] = morph_export_df["Metadata_pdid"].map(pdid_to_drug)
morph_export_df = morph_export_df[
    morph_export_df["Metadata_Drug"].isin(shared_compounds)
].reset_index(drop=True)
morph_export_df["Metadata_moa_fine"] = morph_export_df["Metadata_Drug"].map(moa_by_drug).astype(str)
morph_meta_cols = [c for c in morph_export_df.columns if c.startswith("Metadata_")]
print(
    f"morph_export_df: {len(morph_export_df)} wells, "
    f"{morph_export_df['Metadata_Drug'].nunique()} compounds, "
    f"{morph_export_df['Metadata_Site'].nunique()} sites"
)

119 compounds shared between morphology wells and expression
morph_export_df: 1890 wells, 119 compounds, 4 sites


## 6. Between-plate QC: percent replicating (Way et al. 2022)

Before collapsing replicates into one consensus profile per compound, check
that same-treatment profiles from **different plates** agree more than
chance. Use the Way et al. 2022 **percent replicating** metric (STAR
Methods; Cell Systems) for both modalities:

1. For every treatment with `n >= 2` plate-level replicates, compute the
   **median pairwise Spearman** correlation among those replicates.
2. Build a matched null: draw `n` non-replicate profiles **1,000 times**,
   matching **replicate cardinality**, and for morphology also matching
   **well position** (sample other plate maps at the same well; at most one
   profile per other compound). Expression has no well layout, so the null
   is cardinality-matched only (Way's "relaxed well-position" variant).
3. A treatment "replicates" if its median exceeds the **95th percentile** of
   its null. **Percent replicating** = that fraction of treatments.

Applied here as:

- **Morphology (per site)**: replicates = the up-to-4 wells of the same
  compound across plate replicates `R1`-`R4` (site held fixed; mixing sites
  would confound batch with disagreement).
- **Expression**: one profile per `(drug, plate)` = mean of cell-level PCs
  on that plate; replicates = the same drug across the 14 plates.

Then build morphology consensus profiles (mean over replicate wells) and a
leave-one-out check that the consensus is representative of its inputs.

In [9]:
from itertools import combinations
from scipy.stats import rankdata


def _spearman_corrcoef_rows(X: np.ndarray) -> np.ndarray:
    """Pairwise Spearman correlation between rows (profiles)."""
    ranks = np.apply_along_axis(rankdata, 1, np.asarray(X, dtype=np.float64))
    with np.errstate(invalid="ignore"):
        corr = np.corrcoef(ranks)
    return np.nan_to_num(corr, nan=0.0)


def _median_pairwise(corr: np.ndarray, idxs: list[int] | np.ndarray) -> float:
    idxs = list(idxs)
    return float(np.median([corr[i, j] for i, j in combinations(idxs, 2)]))


def percent_replicating(
    df: pd.DataFrame,
    feature_cols: list[str],
    group_col: str,
    *,
    well_col: str | None = None,
    n_null: int = 1000,
    quantile: float = 0.95,
    seed: int = RANDOM_STATE,
) -> tuple[pd.Series, pd.Series, float]:
    """Way et al. 2022 percent replicating (STAR Methods).

    Median pairwise Spearman among replicates of each treatment, compared to a
    cardinality-matched null (and well-position-matched when ``well_col`` is
    set). A treatment replicates if its median exceeds the ``quantile`` of its
    null. Returns ``(replicate_corr, null_threshold_per_group, pct_replicating)``.
    """
    rng = np.random.RandomState(seed)
    groups = df[group_col].to_numpy()
    wells = df[well_col].to_numpy() if well_col is not None else None
    corr = _spearman_corrcoef_rows(df[feature_cols].to_numpy())

    group_to_idx: dict = {}
    for i, g in enumerate(groups):
        group_to_idx.setdefault(g, []).append(i)

    replicated = {g: idxs for g, idxs in group_to_idx.items() if len(idxs) >= 2}
    replicate_corr = pd.Series(
        {g: _median_pairwise(corr, idxs) for g, idxs in replicated.items()},
        dtype=float,
    )

    # Cardinality -> shared null when wells are not matched.
    cardinality_null: dict[int, np.ndarray] = {}
    if wells is None:
        for k in sorted({len(idxs) for idxs in replicated.values()}):
            null_medians = np.empty(n_null)
            for t in range(n_null):
                cand_groups = list(group_to_idx)
                chosen_groups = rng.choice(cand_groups, size=k, replace=False)
                sample = [rng.choice(group_to_idx[g]) for g in chosen_groups]
                null_medians[t] = _median_pairwise(corr, sample)
            cardinality_null[k] = null_medians

    null_threshold = {}
    for g, idxs in replicated.items():
        k = len(idxs)
        if wells is None:
            null_medians = cardinality_null[k]
        else:
            # Well-matched null: other plate maps at the same well as g,
            # at most one profile per other treatment (Way et al.).
            w = wells[idxs[0]]
            other_by_group: dict = {}
            for i, (gi, wi) in enumerate(zip(groups, wells)):
                if wi == w and gi != g:
                    other_by_group.setdefault(gi, []).append(i)
            cand_groups = list(other_by_group)
            if len(cand_groups) < k:
                # Too few other treatments at this well; fall back to
                # cardinality-matched null without well constraint.
                cand_groups = [gg for gg in group_to_idx if gg != g]
                other_by_group = {gg: group_to_idx[gg] for gg in cand_groups}
            null_medians = np.empty(n_null)
            for t in range(n_null):
                chosen_groups = rng.choice(cand_groups, size=k, replace=False)
                sample = [rng.choice(other_by_group[gg]) for gg in chosen_groups]
                null_medians[t] = _median_pairwise(corr, sample)
        null_threshold[g] = float(np.quantile(null_medians, quantile))

    null_threshold = pd.Series(null_threshold, dtype=float)
    pct_replicating = float((replicate_corr > null_threshold).mean())
    return replicate_corr, null_threshold, pct_replicating


print("Morphology — percent replicating between plate replicates (per site)")
percent_replicating_by_site = {}
replicate_corr_by_site = {}
for site, site_df in morph_export_df.groupby("Metadata_Site"):
    replicate_corr, null_threshold, pct = percent_replicating(
        site_df,
        morph_feature_cols,
        "Metadata_Drug",
        well_col="Metadata_Well",
    )
    percent_replicating_by_site[site] = pct
    replicate_corr_by_site[site] = replicate_corr
    print(
        f"{site}: {pct:.1%} percent replicating "
        f"(median replicate r={replicate_corr.median():.3f}, "
        f"median null 95th={null_threshold.median():.3f}, "
        f"n={len(replicate_corr)} compounds with >=2 replicates)"
    )

Morphology — percent replicating between plate replicates (per site)
FMP_HepG2: 80.5% percent replicating (median replicate r=0.489, median null 95th=0.203, n=118 compounds with >=2 replicates)
IMTM_HepG2: 88.1% percent replicating (median replicate r=0.543, median null 95th=0.199, n=118 compounds with >=2 replicates)
MEDINA_HepG2: 73.1% percent replicating (median replicate r=0.426, median null 95th=0.246, n=119 compounds with >=2 replicates)
USC_HepG2: 80.7% percent replicating (median replicate r=0.491, median null 95th=0.206, n=119 compounds with >=2 replicates)


### Expression - percent replicating between plates

Same metric on gene expression: collapse cells to one profile per
`(drug, plate)` by averaging cell-level PCs, then ask whether a drug's
profiles across plates are more coherent than a cardinality-matched null.
No well-position constraint (this assay has no plate-layout well metadata),
which matches Way's relaxed null variant.

In [10]:
# One expression profile per (drug, plate) from the existing cell-level PCA.
rna_plate_pcs = pd.DataFrame(
    rna.obsm["X_pca"],
    index=rna.obs_names,
    columns=[f"PC{i+1}" for i in range(rna.obsm["X_pca"].shape[1])],
)
rna_plate_pcs["drug"] = rna.obs["drug"].to_numpy()
rna_plate_pcs["plate"] = rna.obs["plate"].to_numpy()

rna_plate_profiles = (
    rna_plate_pcs.groupby(["drug", "plate"], observed=True)
    .mean()
    .reset_index()
)
# Restrict to compounds that also enter the morphology QC / shared set.
rna_plate_profiles = rna_plate_profiles[
    rna_plate_profiles["drug"].isin(shared_compounds)
].reset_index(drop=True)
rna_pc_cols = [c for c in rna_plate_profiles.columns if c.startswith("PC")]

print(
    f"Expression plate profiles: {len(rna_plate_profiles)} (drug, plate) rows, "
    f"{rna_plate_profiles['drug'].nunique()} drugs, "
    f"{rna_plate_profiles['plate'].nunique()} plates"
)
print(
    "Plates per drug:\n"
    + rna_plate_profiles.groupby("drug", observed=True).size()
    .value_counts()
    .sort_index()
    .to_string()
)

rna_replicate_corr, rna_null_threshold, rna_pct_replicating = percent_replicating(
    rna_plate_profiles,
    rna_pc_cols,
    "drug",
    well_col=None,
)
print(
    f"\nExpression: {rna_pct_replicating:.1%} percent replicating "
    f"(median replicate r={rna_replicate_corr.median():.3f}, "
    f"null 95th={rna_null_threshold.median():.3f}, "
    f"n={len(rna_replicate_corr)} drugs with >=2 plates)"
)

Expression plate profiles: 396 (drug, plate) rows, 119 drugs, 14 plates
Plates per drug:
1     3
3    76
4    36
5     3
6     1

Expression: 53.4% percent replicating (median replicate r=0.438, null 95th=0.433, n=116 drugs with >=2 plates)


In [11]:
def leave_one_out_agreement(df: pd.DataFrame, feature_cols: list[str], drug_col: str) -> np.ndarray:
    agreements = []
    for _, group in df.groupby(drug_col):
        X = group[feature_cols].to_numpy()
        if len(X) < 2:
            continue
        for i in range(len(X)):
            loo_mean = (X.sum(axis=0) - X[i]) / (len(X) - 1)
            agreements.append(np.corrcoef(X[i], loo_mean)[0, 1])
    return np.array(agreements)


print("Leave-one-out check: correlation between each replicate well and the mean of its\n"
      "OTHER replicates (excluding itself) - confirms the mean isn't just reproducing one well.\n")
for site, site_df in morph_export_df.groupby("Metadata_Site"):
    agreement = leave_one_out_agreement(site_df, morph_feature_cols, "Metadata_Drug")
    print(f"{site}: mean r={agreement.mean():.3f}, median r={np.median(agreement):.3f} "
          f"(n={len(agreement)} replicate wells)")


Leave-one-out check: correlation between each replicate well and the mean of its
OTHER replicates (excluding itself) - confirms the mean isn't just reproducing one well.

FMP_HepG2: mean r=0.563, median r=0.618 (n=471 replicate wells)
IMTM_HepG2: mean r=0.644, median r=0.711 (n=468 replicate wells)
MEDINA_HepG2: mean r=0.555, median r=0.583 (n=475 replicate wells)
USC_HepG2: mean r=0.571, median r=0.636 (n=476 replicate wells)


In [12]:
consensus_by_site = {}
for site, site_df in morph_export_df.groupby("Metadata_Site"):
    consensus = site_df.groupby("Metadata_Drug")[morph_feature_cols].mean()
    consensus_adata = ad.AnnData(
        X=consensus.to_numpy(dtype=np.float32),
        obs=pd.DataFrame(
            {
                "Metadata_Drug": consensus.index,
                "Metadata_moa_fine": consensus.index.map(moa_by_drug).astype(str),
                "Metadata_Site": site,
                "Metadata_n_replicates": site_df.groupby("Metadata_Drug").size().reindex(consensus.index).to_numpy(),
            },
            index=consensus.index,
        ),
        var=pd.DataFrame(index=morph_feature_cols),
    )
    consensus_adata.write_h5ad(f"morphology_{site}_consensus.h5ad")
    consensus_by_site[site] = consensus_adata
    print(f"morphology_{site}_consensus.h5ad: {consensus_adata.shape} "
          f"({consensus_adata.obs['Metadata_n_replicates'].mean():.1f} replicates/compound on average)")


morphology_FMP_HepG2_consensus.h5ad: (118, 636) (4.0 replicates/compound on average)
morphology_IMTM_HepG2_consensus.h5ad: (118, 636) (4.0 replicates/compound on average)
morphology_MEDINA_HepG2_consensus.h5ad: (119, 636) (4.0 replicates/compound on average)
morphology_USC_HepG2_consensus.h5ad: (119, 636) (4.0 replicates/compound on average)


**Result (re-run cells above to fill numbers):** morphology percent
replicating is computed per site with a well-position-matched Spearman null
(Way et al.); expression percent replicating uses plate-level PC means with a
cardinality-matched null. Leave-one-out below still checks that morphology
consensus profiles are representative of their input replicate wells.

### All-sites consensus

Also build one consensus profile per compound by averaging over *every*
replicate well from *all four* sites at once (up to 16 wells/compound,
instead of up to 4). Unlike the per-site checks above, this deliberately
mixes site/batch variation into the "replicate" average - so re-run the
same two checks on the pooled wells to see how much agreement degrades once
site is no longer held fixed, rather than assuming it is fine.


In [13]:
_, all_sites_null_threshold, all_sites_pct_replicating = percent_replicating(
    morph_export_df,
    morph_feature_cols,
    "Metadata_Drug",
    well_col="Metadata_Well",
)
all_sites_agreement = leave_one_out_agreement(morph_export_df, morph_feature_cols, "Metadata_Drug")

print(f"All sites pooled: {all_sites_pct_replicating:.1%} percent replicating "
      f"(median null 95th r={all_sites_null_threshold.median():.3f})")
print(f"All sites pooled: leave-one-out mean r={all_sites_agreement.mean():.3f}, "
      f"median r={np.median(all_sites_agreement):.3f} (n={len(all_sites_agreement)} wells)")
print(f"\nFor comparison, per-site leave-one-out means were: " +
      ", ".join(f"{s}={leave_one_out_agreement(df, morph_feature_cols, 'Metadata_Drug').mean():.3f}"
                for s, df in morph_export_df.groupby("Metadata_Site")))

All sites pooled: 93.3% percent replicating (median null 95th r=0.060)
All sites pooled: leave-one-out mean r=0.498, median r=0.536 (n=1890 wells)

For comparison, per-site leave-one-out means were: FMP_HepG2=0.563, IMTM_HepG2=0.644, MEDINA_HepG2=0.555, USC_HepG2=0.571


In [14]:
consensus_all_sites = morph_export_df.groupby("Metadata_Drug")[morph_feature_cols].mean()
n_wells_per_drug = morph_export_df.groupby("Metadata_Drug").size().reindex(consensus_all_sites.index)
n_sites_per_drug = morph_export_df.groupby("Metadata_Drug")["Metadata_Site"].nunique().reindex(consensus_all_sites.index)

consensus_all_sites_adata = ad.AnnData(
    X=consensus_all_sites.to_numpy(dtype=np.float32),
    obs=pd.DataFrame(
        {
            "Metadata_Drug": consensus_all_sites.index,
            "Metadata_moa_fine": consensus_all_sites.index.map(moa_by_drug).astype(str),
            "Metadata_n_wells": n_wells_per_drug.to_numpy(),
            "Metadata_n_sites": n_sites_per_drug.to_numpy(),
        },
        index=consensus_all_sites.index,
    ),
    var=pd.DataFrame(index=morph_feature_cols),
)
consensus_all_sites_adata.write_h5ad("morphology_all_sites_consensus.h5ad")

print(f"morphology_all_sites_consensus.h5ad: {consensus_all_sites_adata.shape}, "
      f"{consensus_all_sites_adata.obs['Metadata_n_wells'].mean():.1f} wells/compound on average "
      f"across {consensus_all_sites_adata.obs['Metadata_n_sites'].mean():.2f} sites on average")


morphology_all_sites_consensus.h5ad: (119, 636), 15.9 wells/compound on average across 3.98 sites on average


## 7. Phenotypic activity vs. negative controls (copairs mAP)

Percent replicating (section 6) asks whether a compound's replicate wells
agree with *each other* more than with random other wells - it says nothing
about whether that agreement reflects an actual phenotype versus, say, a
shared plate artifact. The complementary, standard image-based-profiling
check is **mean average precision (mAP) against negative controls**: for
each compound, how well do its replicate wells retrieve each other when
ranked against a pool of untreated DMSO wells, rather than against other
compounds. `copairs` (`cytomining/copairs`, the package behind the JUMP
Cell Painting consortium's activity calls) computes this directly, together
with a permutation null and Benjamini-Hochberg FDR correction per compound.

Restricted, as throughout this notebook, to the 119 `shared_compounds` plus
the recovered DMSO wells (784/site) - not the full ~2464-compound library -
computed per site and on the sites pooled together.

In [15]:
import tqdm.auto
from copairs.map import average_precision, mean_average_precision

tqdm.auto.tqdm = lambda it=None, *a, **k: it if it is not None else iter([])  # copairs' internal find_pairs() ignores progress_bar=False

NULL_SIZE = 10000
P_THRESHOLD = 0.05


def negcon_map(df: pd.DataFrame, feature_cols: list[str], group_col: str) -> pd.DataFrame:
    ap = average_precision(
        meta=df[[group_col, "Metadata_negcon"]],
        feats=df[feature_cols].to_numpy(dtype=np.float64),
        pos_sameby=[group_col], pos_diffby=[],
        neg_sameby=[], neg_diffby=["Metadata_negcon"],
        progress_bar=False,
    )
    ap_trt = ap[~ap["Metadata_negcon"] & (ap["n_pos_pairs"] > 0)]
    return mean_average_precision(
        ap_trt, sameby=[group_col], null_size=NULL_SIZE, threshold=P_THRESHOLD,
        seed=RANDOM_STATE, progress_bar=False,
    )


def negcon_subset(df: pd.DataFrame) -> pd.DataFrame:
    sub = df.copy()
    sub["Metadata_Drug"] = sub["Metadata_pdid"].map(pdid_to_drug)
    sub.loc[sub["Metadata_negcon"], "Metadata_Drug"] = "DMSO"
    return sub[sub["Metadata_negcon"] | sub["Metadata_Drug"].isin(shared_compounds)].reset_index(drop=True)


map_by_site = {}
for site, site_df in hepg2_selected.groupby("Metadata_Site"):
    site_sub = negcon_subset(site_df)
    n_negcon = site_sub["Metadata_negcon"].sum()
    mAP = negcon_map(site_sub, morph_feature_cols, "Metadata_Drug")
    map_by_site[site] = mAP
    pct_active = mAP["below_corrected_p"].mean()
    print(f"{site}: {pct_active:.1%} of compounds phenotypically active vs. DMSO "
          f"(median mAP={mAP['mean_average_precision'].median():.3f}, "
          f"n={len(mAP)} compounds, {n_negcon} DMSO wells)")


FMP_HepG2: 85.6% of compounds phenotypically active vs. DMSO (median mAP=0.917, n=118 compounds, 784 DMSO wells)
IMTM_HepG2: 94.1% of compounds phenotypically active vs. DMSO (median mAP=0.788, n=118 compounds, 784 DMSO wells)
MEDINA_HepG2: 90.8% of compounds phenotypically active vs. DMSO (median mAP=0.600, n=119 compounds, 784 DMSO wells)
USC_HepG2: 89.9% of compounds phenotypically active vs. DMSO (median mAP=0.635, n=119 compounds, 784 DMSO wells)


In [16]:
all_sites_sub = negcon_subset(hepg2_selected)
n_negcon_all = all_sites_sub["Metadata_negcon"].sum()
mAP_all_sites = negcon_map(all_sites_sub, morph_feature_cols, "Metadata_Drug")
pct_active_all_sites = mAP_all_sites["below_corrected_p"].mean()

print(f"All sites pooled: {pct_active_all_sites:.1%} of compounds phenotypically active vs. DMSO "
      f"(median mAP={mAP_all_sites['mean_average_precision'].median():.3f}, "
      f"n={len(mAP_all_sites)} compounds, {n_negcon_all} DMSO wells)")
print("\nFor comparison, per-site:")
for site, mAP in map_by_site.items():
    print(f"  {site}: {mAP['below_corrected_p'].mean():.1%} active, "
          f"median mAP={mAP['mean_average_precision'].median():.3f}")


All sites pooled: 95.0% of compounds phenotypically active vs. DMSO (median mAP=0.327, n=119 compounds, 3136 DMSO wells)

For comparison, per-site:
  FMP_HepG2: 85.6% active, median mAP=0.917
  IMTM_HepG2: 94.1% active, median mAP=0.788
  MEDINA_HepG2: 90.8% active, median mAP=0.600
  USC_HepG2: 89.9% active, median mAP=0.635


**Result**: 86-95% of compounds are phenotypically active vs. DMSO at
each individual site (FMP 85.6%, IMTM 94.1%, MEDINA 90.8%, USC 89.9%), and
95.0% when all sites are pooled (784 DMSO wells/site, 3136 pooled) - a
similar range to percent replicating (68-89% per site, 94.1% pooled, section
9), confirming most of these bioactive-library compounds produce a real,
statistically distinguishable morphological phenotype relative to untreated
cells, not just internal replicate consistency. Pooling sites raises the
active fraction further (more DMSO wells sharpen the permutation null) even
though median mAP itself *drops* when pooled (0.327 vs. 0.600-0.917 per
site) - site is still a real batch effect diluting the raw similarity
signal, it just doesn't cost many compounds their significance call once
there's enough negative-control data to detect a phenotype against.

## 8. QC summary

- Shared set: compounds with both Tahoe expression cells and HepG2 Cell
  Painting wells after feature selection (typically 119; `Trametinib` is the
  known morphology gap).
- **Percent replicating** (section 6, Way et al. 2022): between-plate
  reproducibility for morphology (per site, well-position-matched Spearman
  null) and expression (`drug x plate` PC means, cardinality-matched null).
  Re-run section 6 for numeric rates.
- Morphology **consensus** profiles (mean over plate-replicate wells) are
  written per site and pooled; leave-one-out checks that the mean is
  representative of its inputs.
- **copairs mAP vs. DMSO** (section 7): fraction of shared compounds with a
  significant morphological phenotype against negative controls. Re-run for
  numeric rates.

Cross-modal Mantel / MoA silhouette / joint UMAP live in
`morphology_expression_integration.ipynb`.